In [ ]:
# Portable project paths. Set TLS_PROJECT_ROOT to the directory containing the input data.
import os
from pathlib import Path
PROJECT_ROOT = Path(os.environ.get("TLS_PROJECT_ROOT", ".")).resolve()


# Supplementary Figure 33 plotting code


## Shared setup


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Polygon, Rectangle, Circle, FancyArrowPatch
from scipy.spatial import ConvexHull, QhullError

ROOT = Path.cwd()
DATA = ROOT / "source_data"
OUT = ROOT / "output"
OUT.mkdir(exist_ok=True)
assert DATA.exists(), "Run this notebook from the Figure 6 code directory."

mpl.rcParams.update({
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "svg.fonttype": "none",
    "font.family": "DejaVu Sans",
    "font.size": 7,
    "axes.titlesize": 8,
    "axes.labelsize": 7,
    "xtick.labelsize": 6,
    "ytick.labelsize": 6,
    "axes.linewidth": 0.6,
    "xtick.major.width": 0.5,
    "ytick.major.width": 0.5,
    "legend.frameon": False,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

blue = "#2B5C96"
red = "#C94C4C"
grey = "#8B9AA7"
dark = "#152536"

cell_colors = {
    "Epithelial": "#1F7A7A", "Fibroblast": "#E59B3A", "Endothelial": "#39A56A",
    "Pericyte": "#93B5D7", "T cell": "#E64B35", "NK cell": "#7E57C2",
    "B cell": "#2C7FB8", "Plasma cell": "#4EA3D8", "Monocyte/Macrophage": "#61C2A2",
    "Dendritic cell": "#D94F9B", "Mast cell": "#8CD36B", "Neutrophil": "#37BDBD",
    "Mix": "#C9C9C9", "Unknown": "#D9D9D9",
    "T/NK cell": "#4FA366", "Myeloid": "#EF6C42", "Fibroblast/FDC": "#F1A55C",
    "Endothelial/Pericyte": "#5EB4A5", "Other/Unknown": "#BDBDBD", "Other immune": "#BDBDBD",
    "B cells": "#2C7FB8", "DC": "#D94F9B", "DC cells": "#D94F9B",
    "Endothelial cells": "#39A56A", "Epithelial cells": "#1F7A7A",
    "Fibroblasts": "#E59B3A", "Macrophages": "#61C2A2",
    "Mast cells": "#8CD36B", "Monocytes": "#61C2A2",
    "T cells": "#E64B35", "NK cells": "#7E57C2"
}

class_colors = {"Conforming TLS": blue, "Deviating TLS": red, "Mature TLS": "#2B7A3D"}

from matplotlib.gridspec import GridSpec
def savefig(fig, name):
    for ext in ["pdf", "svg", "png"]:
        fig.savefig(OUT / f"{name}.{ext}", dpi=450, bbox_inches="tight")

def sem(x):
    x = pd.Series(x).dropna()
    return x.std(ddof=1) / np.sqrt(len(x)) if len(x) > 1 else np.nan

def draw_hull(ax, df, x="X", y="Y", color="k", lw=0.55, alpha=1, pad=0):
    pts = df[[x, y]].dropna().drop_duplicates().to_numpy()
    if len(pts) < 3:
        return
    try:
        h = ConvexHull(pts)
        poly = Polygon(pts[h.vertices], closed=True, fill=False, ec=color, lw=lw, alpha=alpha, joinstyle="round")
        ax.add_patch(poly)
    except QhullError:
        pass

def add_panel(ax, label):
    ax.text(-0.08, 1.05, label, transform=ax.transAxes, ha="left", va="bottom", fontsize=9, fontweight="bold")


## Supplementary Fig. 33a


In [ ]:
def panel_a(ax):
    desi = pd.read_csv(DATA / "SourceData_Fig6_DESI_EPAS1high_vs_low_features.csv")
    sel = desi[desi["selected_lipid_program_feature"] == True].sort_values("delta_z_high_minus_low")
    y = np.arange(len(sel))
    ax.barh(y, sel["delta_z_high_minus_low"], color=red, alpha=0.85)
    ax.set_yticks(y); ax.set_yticklabels(sel["short_label"], fontsize=6)
    ax.set_xlabel("EPAS1 coefficient")
    ax.set_title("Individual lipid features", loc="left")
    for yi, p in zip(y, sel["p_welch"]):
        if p < 0.05:
            ax.text(sel["delta_z_high_minus_low"].iloc[yi] + 0.005, yi, "*", va="center")


## Supplementary Fig. 33b


In [ ]:
def panel_b(ax_main, ax_eff):
    scores = pd.read_csv(DATA / "SourceData_Fig6_periTLS_epithelial_RNA_proxy_scores.csv")
    genes = pd.read_csv(DATA / "SourceData_Fig6_periTLS_epithelial_RNA_proxy_gene_sets.csv")
    metrics = ["Hypoxia / HIF epithelial proxy", "Glycolysis epithelial proxy", "OXPHOS / ETC epithelial RNA proxy", "TCA / mitochondrial metabolism proxy"]
    label_map = {"Hypoxia / HIF epithelial proxy":"Hypoxia / HIF", "Glycolysis epithelial proxy":"Glycolysis", "OXPHOS / ETC epithelial RNA proxy":"OXPHOS / ETC", "TCA / mitochondrial metabolism proxy":"TCA / mito."}
    rows = []
    for m in metrics:
        for cls, sub in scores[scores["tls_class"].isin(["Conforming TLS", "Deviating TLS"])].groupby("tls_class"):
            for v in sub[m].dropna():
                rows.append((label_map[m], cls, v))
    long = pd.DataFrame(rows, columns=["score", "tls_class", "value"])
    ypos = np.arange(len(metrics))[::-1]
    for i, m in enumerate([label_map[x] for x in metrics]):
        for cls, c, off in [("Conforming TLS", blue, -0.12), ("Deviating TLS", red, 0.12)]:
            v = long[(long["score"] == m) & (long["tls_class"] == cls)]["value"]
            ax_main.scatter(v, np.full(len(v), ypos[i] + off), s=5, c=c, alpha=0.25, lw=0, rasterized=True)
            ax_main.errorbar(v.mean(), ypos[i] + off, xerr=v.std(), fmt="o", c=c, ms=3, lw=0.7, capsize=2)
    ax_main.axvline(0, color="#AAB7C2", lw=0.6, ls="--")
    ax_main.set_yticks(ypos); ax_main.set_yticklabels([label_map[x] for x in metrics])
    ax_main.set_xlabel("peri-TLS epithelial RNA proxy z score")
    ax_main.set_title("Peri-TLS epithelial oxygen-demand programs", loc="left")
    eff = genes.drop_duplicates("score").set_index("score").loc[metrics]
    ax_eff.errorbar(eff["sample_adjusted_beta"], ypos, xerr=0, fmt="o", c=red, ms=3)
    for y, (_, r) in zip(ypos, eff.iterrows()):
        ax_eff.text(r["sample_adjusted_beta"] + 0.02, y, f"+{r['sample_adjusted_beta']:.2f}\nFDR={r['sample_adjusted_fdr_bh']:.3g}", color=red, va="center", fontsize=5)
    ax_eff.axvline(0, color="#AAB7C2", lw=0.6)
    ax_eff.set_yticks([]); ax_eff.set_xlabel("Adjusted beta\n(Dev - Conf)")


## Supplementary Fig. 33c


In [ ]:
def panel_c(ax):
    df = pd.read_csv(DATA / "SourceData_Fig6_celltype_lipid_attribution.csv")
    df = df[df["tls_class"].isin(["Conforming TLS", "Deviating TLS"])]
    order = ["Endothelial/Pericyte", "T/NK cell", "B cell", "Myeloid", "Fibroblast/FDC"]
    offsets = {"Conforming TLS": -0.18, "Deviating TLS": 0.18}
    for i, ct in enumerate(order):
        for cls, c in [("Conforming TLS", blue), ("Deviating TLS", red)]:
            vals = df[(df["broad_celltype"] == ct) & (df["tls_class"] == cls)]["mean_lipid_program_z"].dropna()
            bp = ax.boxplot([vals], positions=[i+offsets[cls]], widths=0.28, showfliers=False, patch_artist=True,
                            medianprops=dict(color=dark, lw=0.6), boxprops=dict(color=c, lw=0.5), whiskerprops=dict(color=c, lw=0.5), capprops=dict(color=c, lw=0.5))
            bp["boxes"][0].set_facecolor(c); bp["boxes"][0].set_alpha(0.18)
            ax.scatter(np.random.default_rng(i).normal(i+offsets[cls], 0.03, len(vals)), vals, s=4, c=c, alpha=0.25, lw=0, rasterized=True)
    ax.axhline(0, color="#AAB7C2", lw=0.6, ls="--")
    ax.set_xticks(range(len(order))); ax.set_xticklabels(order, rotation=35, ha="right")
    ax.set_ylabel("spot DESI lipid")
    ax.set_title("Spot-level DESI lipid signal by annotated cell type", loc="left")
